# Round 2: ask the VLM to *choose*, not to measure

Round 1 asked Qwen2-VL-2B to read a dimension off the photo. It scored **11.0%**
against the cascade's 36.2% — because it answered `"unknown"` on 79% of rows.
But when it did commit it was **51.8%** precise, against the cascade's 40.7%.
So the signal is there; the task framing was wrong.

The measured bottleneck was never *reading* — OCR already finds a matching
number **70.1%** of the time on these rows. The bottleneck is deciding **which**
number is the width. So this round hands the model the OCR candidates and asks
it to pick one. Open-ended measurement becomes multiple choice.

Two variants, same 398 rows:

| Variant | Prompt |
|---|---|
| **A · forced** | the round-1 question with the `"unknown"` escape removed |
| **B · select** | "OCR read these values — which is the width?" |

Rows with **0** candidates abstain without a model call; rows with exactly **1**
take it (that is what the rules already do). The model is only asked about the
**271 rows with 2+ candidates** — the actual decision.

| Reference | dim |
|---|---|
| VLM round 1, open-ended | 11.0% |
| S0 rules over OCR | 31.8% |
| tagger + rules cascade | **36.2%** ← the bar |
| gold is among the candidates | 70.1% ← selection ceiling |

**Setup:** Accelerator `GPU T4 x2`, Internet **On**, and attach the dataset
containing `vlm_select_sample.tar.gz`.

In [ ]:
import glob, os, subprocess, sys, tarfile

found = (glob.glob("/kaggle/input/**/sample.csv", recursive=True)
         + glob.glob("**/sample.csv", recursive=True))
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
if found:
    DEST = os.path.dirname(os.path.abspath(found[0]))
    print("using extracted data:", DEST)
else:
    tars = (glob.glob("/kaggle/input/**/vlm_select_sample.tar.gz", recursive=True)
            + glob.glob("**/vlm_select_sample.tar.gz", recursive=True))
    if not tars:
        raise SystemExit("No sample.csv or vlm_select_sample.tar.gz under /kaggle/input.")
    with tarfile.open(tars[0]) as t:
        t.extractall(WORK)
    DEST = os.path.join(WORK, "vlm_select")

assert os.path.isfile(f"{DEST}/sample.csv") and os.path.isdir(f"{DEST}/images")
print("images:", len(os.listdir(f"{DEST}/images")))

import torch
print("cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if not torch.cuda.is_available():
    print("\n!! Settings panel > Accelerator > GPU T4 x2, then re-run.")

import transformers, packaging.version as V
if V.parse(transformers.__version__) < V.parse("4.45"):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U",
                    "transformers>=4.45", "accelerate"], check=True)
    print("\n*** upgraded transformers - now do Run > Restart & run all ***")
else:
    print("transformers", transformers.__version__, "ok")

In [ ]:
import csv, json, re, time, string
from collections import Counter, defaultdict
from PIL import Image

sys.path.insert(0, DEST)
import normalize as N

MODEL = "Qwen/Qwen2-VL-2B-Instruct"
PHRASE = {"depth": "depth (front to back)",
          "width": "width (left to right)",
          "height": "height (top to bottom)"}

FORCED = (
    "This is a product photograph, which may include printed dimension labels or arrows.\n"
    "What is the product's {phrase}?\n"
    'You must give a best estimate. Answer with only a number and a unit, e.g. "12 cm".'
)

SELECT = (
    "This product photograph shows measurements. Text recognition read these "
    "values from the image:\n{options}\n"
    "Which one is the product's {phrase}?\n"
    "Answer with only the letter."
)

_ANS = re.compile(r"(-?\d+(?:[.,]\d+)?)\s*([A-Za-z\"'.]+)")

def parse_value(raw):
    if not raw or "unknown" in raw.lower():
        return None
    m = _ANS.search(raw.replace("\u2033", "in").replace("\u201d", "in"))
    if not m:
        return None
    u = N.normalize_unit(m.group(2).strip("\"'."))
    if u is None or N.family(u) != "length":
        return None
    return float(m.group(1).replace(",", ".")), u

def parse_letter(raw, k):
    """First a-h letter in the reply, bounded by the option count."""
    m = re.search(r"\b([a-h])\b", (raw or "").lower())
    if not m:
        return None
    i = string.ascii_lowercase.index(m.group(1))
    return i if i < k else None

def dim_correct(pred, lo, hi, gu, tol=0.01):
    if pred is None:
        return False
    v, u = pred
    if N.family(u) != N.family(gu):
        return False
    a = N.to_base(v, u)
    return N.to_base(lo, gu)*(1-tol) <= a <= N.to_base(hi, gu)*(1+tol)

def parse_cands(s):
    out = []
    for part in (s or "").split(";"):
        part = part.strip()
        if not part:
            continue
        p = parse_value(part)
        if p:
            out.append(p)
    return out

rows = list(csv.DictReader(open(f"{DEST}/sample.csv")))
for r in rows:
    r["cands"] = parse_cands(r["candidates"])
print(f"{len(rows)} rows | 0 cands: {sum(1 for r in rows if not r['cands'])}"
      f" | 1: {sum(1 for r in rows if len(r['cands'])==1)}"
      f" | 2+: {sum(1 for r in rows if len(r['cands'])>=2)}")

In [ ]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
dev = "cuda" if torch.cuda.is_available() else "cpu"
proc = AutoProcessor.from_pretrained(MODEL, min_pixels=256*28*28, max_pixels=768*768)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL, dtype=torch.float16).to(dev).eval()

def ask(img, prompt, max_new=16):
    msgs = [{"role": "user", "content": [{"type": "image"},
                                         {"type": "text", "text": prompt}]}]
    text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = proc(text=[text], images=[img], return_tensors="pt").to(dev)
    with torch.no_grad():
        gen = model.generate(**inp, max_new_tokens=max_new, do_sample=False)
    return proc.batch_decode(gen[:, inp["input_ids"].shape[1]:],
                             skip_special_tokens=True)[0].strip()

print("model ready on", dev)

In [ ]:
res = defaultdict(list)
t0 = time.time()
for i, r in enumerate(rows):
    img = Image.open(f"{DEST}/images/{r['image_file']}").convert("RGB")
    a, lo, hi, gu = r["attribute"], float(r["gold_lo"]), float(r["gold_hi"]), r["gold_unit"]
    cands = r["cands"]

    # --- A: forced open-ended answer ---
    raw_a = ask(img, FORCED.format(phrase=PHRASE[a]))
    pred_a = parse_value(raw_a)

    # --- B: selection among OCR candidates ---
    if not cands:
        pred_b, raw_b = None, "(no candidates)"
    elif len(cands) == 1:
        pred_b, raw_b = cands[0], "(single candidate)"
    else:
        opts = "\n".join(f"  ({string.ascii_lowercase[j]}) {v:g} {u}"
                          for j, (v, u) in enumerate(cands))
        raw_b = ask(img, SELECT.format(options=opts, phrase=PHRASE[a]), max_new=8)
        j = parse_letter(raw_b, len(cands))
        pred_b = cands[j] if j is not None else None

    res["rows"].append({
        "image_file": r["image_file"], "attribute": a, "n_cands": len(cands),
        "gold": [lo, hi, gu], "gold_in_cands": int(r["gold_in_candidates"]),
        "forced_raw": raw_a, "forced_pred": list(pred_a) if pred_a else None,
        "forced_dim": bool(dim_correct(pred_a, lo, hi, gu)),
        "select_raw": raw_b, "select_pred": list(pred_b) if pred_b else None,
        "select_dim": bool(dim_correct(pred_b, lo, hi, gu)),
    })
    if (i + 1) % 50 == 0:
        d = res["rows"]
        print(f"  {i+1}/{len(rows)}  {(time.time()-t0)/(i+1):.2f}s/row  "
              f"forced {100*sum(x['forced_dim'] for x in d)/len(d):.1f}%  "
              f"select {100*sum(x['select_dim'] for x in d)/len(d):.1f}%", flush=True)

OUT = f"{WORK}/vlm_select_results.json"
json.dump(res["rows"], open(OUT, "w"), indent=1)
print("\nwrote", OUT)

In [ ]:
import math
d = res["rows"]; n = len(d)
def wilson(k, N, z=1.96):
    if N == 0: return (0.0, 0.0)
    p=k/N; den=1+z*z/N; c=p+z*z/(2*N); h=z*math.sqrt(p*(1-p)/N+z*z/(4*N*N))
    return ((c-h)/den, (c+h)/den)

def line(name, key, subset=None):
    sel = [x for x in d if subset is None or subset(x)]
    k = sum(x[key] for x in sel); m = len(sel) or 1
    ans = sum(x[key.replace("_dim", "_pred")] is not None for x in sel)
    lo, hi = wilson(k, m)
    print(f"{name:<34}{m:>5}{100*ans/m:>10.1f}%{100*k/m:>8.1f}%"
          f"{f'[{100*lo:.1f}, {100*hi:.1f}]':>16}")

print(f"{'variant':<34}{'n':>5}{'coverage':>10}{'dim':>8}{'95% CI':>16}")
print("-" * 73)
line("A - forced open-ended", "forced_dim")
line("B - select from candidates", "select_dim")
print("-" * 73)
print(f"{'VLM round 1 (open, may abstain)':<34}{400:>5}{21.2:>9.1f}%{11.0:>8.1f}%")
print(f"{'S0 rules over OCR':<34}{'':>5}{'':>10}{31.8:>8.1f}%")
print(f"{'tagger + rules cascade':<34}{'':>5}{'':>10}{36.2:>8.1f}%")
print(f"{'gold among candidates (ceiling)':<34}{'':>5}{'':>10}{70.1:>8.1f}%")

print(f"\n--- B restricted to the rows the model actually decided (2+ candidates) ---")
print(f"{'variant':<34}{'n':>5}{'coverage':>10}{'dim':>8}{'95% CI':>16}")
print("-" * 73)
line("B - select, 2+ candidates", "select_dim", lambda x: x["n_cands"] >= 2)
line("B - select, gold was present", "select_dim",
     lambda x: x["n_cands"] >= 2 and x["gold_in_cands"])

print(f"\n{'attribute':<10}{'n':>5}{'forced':>9}{'select':>9}")
print("-" * 33)
per = defaultdict(Counter)
for x in d:
    c = per[x["attribute"]]; c["n"] += 1; c["f"] += x["forced_dim"]; c["s"] += x["select_dim"]
for a in ("depth", "width", "height"):
    c = per[a]
    print(f"{a:<10}{c['n']:>5}{100*c['f']/c['n']:>8.1f}%{100*c['s']/c['n']:>8.1f}%")

print("\nDownload vlm_select_results.json from the Output panel and send it back.")